# Module 08: Data Modeling with statsmodels

Follow along here during class. Feel free to add your own cells and notes.

The git section at the start of today's class (`git pull`, `git rebase`, installing `formulaic`) happens in your terminal, not in this notebook.

In [ ]:
# Only needed if you run this notebook in the browser (JupyterLite).
%pip install -q formulaic

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('data/gapminder.tsv', sep='\t')
df = df.rename(columns={'lifeExp': 'life_expectancy', 'gdpPercap': 'gdp_per_capita'})
df['log_gdp'] = np.log10(df['gdp_per_capita'])
df['log_pop'] = np.log10(df['pop'])
df_2007 = df.query("year == 2007")

## statsmodels

### From Correlation to Regression

In [ ]:
import statsmodels.api as sm

In [ ]:
sns.regplot(data=df_2007, x='log_gdp', y='life_expectancy')

### Formulas with formulaic

`~` reads as "is modeled by". `model_matrix()` returns the outcome `y` and the design matrix `X`.

In [ ]:
from formulaic import model_matrix

y, X = model_matrix('life_expectancy ~ log_gdp', df_2007)
X.head()

### Fitting OLS

In [ ]:
model = sm.OLS(y, X)
results = model.fit()
print(results.summary())

### Reading the Summary

What is the slope? What does it mean for a 10x increase in GDP per capita? What is the R-squared?

### Adding a Predictor

In [ ]:
y, X = model_matrix('life_expectancy ~ log_gdp + C(continent)', df_2007)
results = sm.OLS(y, X).fit()
print(results.summary())

### Practice (5 min)

Let's work on these questions:

1. Fit `life_expectancy ~ log_gdp` on `df_1952 = df.query("year == 1952")`. Is the slope bigger or smaller than in 2007?
2. Fit `life_expectancy ~ log_gdp + log_pop` on `df_2007`. Is the `log_pop` coefficient significant?
3. What is the `R-squared` of each model?

In [ ]:
# 1.

In [ ]:
# 2.

In [ ]:
# 3.

## Logistic Regression

### Classification

Does a country have a high life expectancy, meaning above the median in 2007?

In [ ]:
median = df_2007['life_expectancy'].median()
df_2007 = df_2007.assign(high_life_exp=(df_2007['life_expectancy'] > median).astype(int))
df_2007['high_life_exp'].value_counts()

### Logistic Regression

In [ ]:
y, X = model_matrix('high_life_exp ~ log_gdp', df_2007)
results = sm.Logit(y, X).fit()
print(results.summary())

### Reading the Logit Summary

What does the sign of the `log_gdp` coefficient tell us? Turn it into an odds ratio:

In [ ]:
np.exp(results.params['log_gdp'])

### Predicted Probabilities

In [ ]:
new = pd.DataFrame({'Intercept': 1, 'log_gdp': [3, 3.5, 4]})
results.predict(new)

### Checking the Predictions

In [ ]:
predicted = (results.predict(X) > 0.5).astype(int)
(predicted == df_2007['high_life_exp']).mean()

In [ ]:
pd.crosstab(df_2007['high_life_exp'], predicted)

### Practice (5 min)

Let's work on these questions:

1. Fit `high_life_exp ~ log_gdp + log_pop`. Is the `log_pop` coefficient significant?
2. Make a column `africa` that is `1` for countries in Africa and `0` otherwise, and fit `africa ~ log_gdp`. Is the `log_gdp` coefficient positive or negative? What does that mean?
3. Use `results.predict()` to find the probability of a high life expectancy for a country with a GDP per capita of $5,000. Hint: use `np.log10(5000)`.

In [ ]:
# 1.

In [ ]:
# 2.

In [ ]:
# 3.

### Practice (10 min)

Let's work on these questions:

1. Fit a statsmodels OLS of `life_expectancy ~ log_gdp + C(continent)` on 1952. Does continent explain more or less than in 2007?
2. Make `high_life_exp` for 1952, using the 1952 median, and fit `high_life_exp ~ log_gdp`. Is the `log_gdp` coefficient bigger or smaller than in 2007? What is the accuracy?
3. In your own words: which model would you use to answer "how many more years of life expectancy?", and which for "is this country in the high group?"

In [ ]:
# 1.

In [ ]:
# 2.

In [ ]:
# 3.